# 手撕 WordPiece

## 背景
WordPiece 被 BERT 使用，类似 BPE 但用 ## 前缀表示子词。
编码时用贪心最长匹配（greedy longest-match-first）。

## 考察点
- ## 前缀的含义（非词首子词）
- 贪心最长匹配编码
- 与 BPE 的区别

In [ ]:
from collections import Counter

class WordPiece:
    def __init__(self, vocab=None, unk_token="[UNK]"):
        self.vocab = vocab or set()
        self.unk = unk_token

    def train(self, texts, vocab_size=100):
        # 简化训练：统计字符和子词频率
        char_freqs = Counter()
        for text in texts:
            for word in text.split():
                chars = list(word)
                char_freqs[chars[0]] += 1
                for c in chars[1:]:
                    char_freqs[f"##{c}"] += 1
                char_freqs["".join(chars)] += 1  # 完整词
        # 取 top-k 作为词表
        self.vocab = {tok for tok, _ in char_freqs.most_common(vocab_size)}
        self.vocab.add(self.unk)

    def encode_word(self, word):
        # 贪心最长匹配
        tokens = []
        start = 0
        while start < len(word):
            end = len(word)
            cur = None
            while start < end:
                sub = word[start:end]
                candidate = sub if start == 0 else f"##{sub}"
                if candidate in self.vocab:
                    cur = candidate
                    break
                end -= 1
            if cur is None:
                return [self.unk]
            tokens.append(cur)
            start = end
        return tokens

    def encode(self, text):
        tokens = []
        for word in text.split():
            tokens.extend(self.encode_word(word))
        return tokens

In [ ]:
# 验证 WordPiece
texts = ["hello world", "hello there", "world peace"]
wp = WordPiece()
wp.train(texts, vocab_size=50)
print(f"词表大小: {len(wp.vocab)}")
encoded = wp.encode("hello world")
print(f"encode('hello world'): {encoded}")
assert len(encoded) > 0
# 验证 ## 前缀逻辑
if len(encoded) >= 2:
    non_first = [t for t in encoded[1:] if t.startswith("##")]
    print(f"非词首 token 使用 ## 前缀: {non_first}")
print("✅ WordPiece 训练 + 编码验证通过")